# 02 - Data Cleaning

This notebook cleans the DVF data loaded from `data/raw`. The cleaning workflow checks missing values, removes columns with very high missingness, converts important data types, removes invalid rows, removes duplicates, and saves a clean sample for later EDA.

In [ ]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

files = sorted(RAW_DIR.glob("*.txt.zip"))
print("Raw directory:", RAW_DIR)
print("Files found:", len(files))
files[:5]

In [ ]:
if not files:
    raise FileNotFoundError("No .txt.zip files found in data/raw folder")

dfs = []

for file in files:
    print("Loading:", file.name)
    temp_df = pd.read_csv(file, sep="|", low_memory=False)
    temp_df["source_file"] = file.name
    dfs.append(temp_df)

df = pd.concat(dfs, ignore_index=True)
print("Original shape:", df.shape)
df.head()

## Missing Value Analysis

Columns with more than 80% missing values are removed because they have limited analytical value for this project.

In [ ]:
missing_report = pd.DataFrame({
    "missing_count": df.isnull().sum(),
    "missing_percentage": (df.isnull().sum() / len(df)) * 100
}).sort_values(by="missing_percentage", ascending=False)

missing_report.head(20)

In [ ]:
columns_to_drop = missing_report[missing_report["missing_percentage"] > 80].index
print("Columns dropped:")
print(list(columns_to_drop))

df_clean = df.drop(columns=columns_to_drop)
print("Cleaned shape after dropping sparse columns:", df_clean.shape)

## Data Type Conversion

`Date mutation` is converted to a datetime column. `Valeur fonciere` is converted from the French decimal format into a numeric value.

In [ ]:
df_clean["Date mutation"] = pd.to_datetime(
    df_clean["Date mutation"],
    format="%d/%m/%Y",
    errors="coerce"
)

df_clean["Valeur fonciere"] = (
    df_clean["Valeur fonciere"]
    .astype(str)
    .str.replace(",", ".", regex=False)
)

df_clean["Valeur fonciere"] = pd.to_numeric(
    df_clean["Valeur fonciere"],
    errors="coerce"
)

df_clean[["Date mutation", "Valeur fonciere"]].head()

## Remove Invalid Rows And Duplicates

In [ ]:
df_clean = df_clean.dropna(subset=["Date mutation", "Valeur fonciere"])
df_clean = df_clean.drop_duplicates()

print("Shape after date/price conversion and duplicate removal:", df_clean.shape)
df_clean.head()

## Save Clean Sample

A clean sample is saved for downstream notebooks so EDA and feature engineering can run faster.

In [ ]:
sample_path = PROCESSED_DIR / "dvf_clean_sample.csv"
df_clean.head(100000).to_csv(sample_path, index=False)

print("Saved cleaned sample to:")
print(sample_path)